### Build races dimension
1. Read "silver" races table
2. Read "silver" circuit table
3. Join data from races with circuits using circuits_id
4. Select required columns
    -   races.season
    -   races.round
    -   races.race_name
    -   races.race_date
    -   circuits.circuit_name
    -   circuits.locality
    -   circuits.country
7. Write transformed data to gold dim_races table

In [0]:
%run ../Workspace/common/configuration_environment

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

##### 1. Read source table
    a)   races
    b)   circuit



In [0]:
circuits_df = spark.table(f"{catalog_name}.{silver_schema}.circuits")
races_df = spark.table(f"{catalog_name}.{silver_schema}.races")


In [0]:
display(circuits_df)
display(races_df)

circuit_id,circuit_name,latitude,longitude,locality,country_name,ingestion_timestamp,source_file
adelaide,Adelaide Street Circuit,-34.9272,138.617,Adelaide,Australia,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,Ain Diab,33.5786,-7.6875,Casablanca,Morocco,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,Aintree,53.4769,-2.94056,Liverpool,UK,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,Circuit Of The Americas,30.1328,-97.6411,Austin,USA,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,Scandinavian Raceway,57.2653,13.6042,Anderstorp,Sweden,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,Avus,52.4806,13.2514,Berlin,Germany,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
baku,Baku City Circuit,40.3725,49.8533,Baku,Azerbaijan,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
boavista,Circuito Da Boavista,41.1705,-8.67325,Oporto,Portugal,2026-04-27T09:26:29.325242Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


season,round,url,race_name,date,circuit_id,ingestion_timestamp,source_file
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,British Grand Prix,1950-05-13,silverstone,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,Monaco Grand Prix,1950-05-21,monaco,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,Indianapolis 500,1950-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,Swiss Grand Prix,1950-06-04,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,Belgian Grand Prix,1950-06-18,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,French Grand Prix,1950-07-02,reims,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,Italian Grand Prix,1950-09-03,monza,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,Swiss Grand Prix,1951-05-27,bremgarten,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,Indianapolis 500,1951-05-30,indianapolis,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,Belgian Grand Prix,1951-06-17,spa,2026-04-27T09:39:24.920055Z,dbfs:/Volumes/formula1/landing/files/races.csv


#### 2. Join data from races with circuits using circuits_id
Select required columns
1. races.season
2. races.round
3. races.race_name
4. races.race_date
5. circuits.circuit_name
6. circuits.locality
7. circuits.country

In [0]:
dim_races_df = (
    races_df
        .join(circuits_df, 
              races_df.circuit_id == circuits_df.circuit_id, "inner")
        .select(
            races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country_name
        )
)

display(dim_races_df)

season,round,race_name,date,circuit_name,locality,country_name
1950,1,British Grand Prix,1950-05-13,Silverstone Circuit,Silverstone,UK
1950,2,Monaco Grand Prix,1950-05-21,Circuit De Monaco,Monte Carlo,Monaco
1950,3,Indianapolis 500,1950-05-30,Indianapolis Motor Speedway,Indianapolis,USA
1950,4,Swiss Grand Prix,1950-06-04,Circuit Bremgarten,Bern,Switzerland
1950,5,Belgian Grand Prix,1950-06-18,Circuit De Spa-francorchamps,Spa,Belgium
1950,6,French Grand Prix,1950-07-02,Reims-gueux,Reims,France
1950,7,Italian Grand Prix,1950-09-03,Autodromo Nazionale Di Monza,Monza,Italy
1951,1,Swiss Grand Prix,1951-05-27,Circuit Bremgarten,Bern,Switzerland
1951,2,Indianapolis 500,1951-05-30,Indianapolis Motor Speedway,Indianapolis,USA
1951,3,Belgian Grand Prix,1951-06-17,Circuit De Spa-francorchamps,Spa,Belgium


##### 7. Write transformed data to gold dim_races table

In [0]:
(
    dim_races_df
        .write
        .format("delta")
        .saveAsTable(target_table)    
)

#table have been created, can't overwrite it

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7159083415922426>, line 5
      1 (
      2     dim_races_df
      3         .write
      4         .format("delta")
----> 5         .saveAsTable(target_table)    
      6 )
      8 display(spark.table(target_table))

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    219         accessor=wrapper,
    220         module_name=module_name,
   (...)
    224         duration=time.perf_counter() - start
    225     )
    226     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:1876, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
   1874 if format is not

In [0]:
display(spark.table(target_table))

season,round,race_name,date,circuit_name,locality,country_name
1950,1,British Grand Prix,1950-05-13,Silverstone Circuit,Silverstone,UK
1950,2,Monaco Grand Prix,1950-05-21,Circuit De Monaco,Monte Carlo,Monaco
1950,3,Indianapolis 500,1950-05-30,Indianapolis Motor Speedway,Indianapolis,USA
1950,4,Swiss Grand Prix,1950-06-04,Circuit Bremgarten,Bern,Switzerland
1950,5,Belgian Grand Prix,1950-06-18,Circuit De Spa-francorchamps,Spa,Belgium
1950,6,French Grand Prix,1950-07-02,Reims-gueux,Reims,France
1950,7,Italian Grand Prix,1950-09-03,Autodromo Nazionale Di Monza,Monza,Italy
1951,1,Swiss Grand Prix,1951-05-27,Circuit Bremgarten,Bern,Switzerland
1951,2,Indianapolis 500,1951-05-30,Indianapolis Motor Speedway,Indianapolis,USA
1951,3,Belgian Grand Prix,1951-06-17,Circuit De Spa-francorchamps,Spa,Belgium
